
# 🤖 MGMT 467 - Unit 2 Lab 2: Prompt Studio — Feature Engineering & Beyond

**Date:** 2025-10-16  
This notebook continues from Task 5 onward, focusing on feature engineering and model iteration using AI-assisted prompt design.

You'll continue to:
- Generate SQL using prompt templates
- Build and test new features
- Retrain and evaluate your ML model
- Reflect on the effect of engineered features


In [9]:
from google.colab import auth
from google.cloud import bigquery

# Authenticate with Google
auth.authenticate_user()

# Set your BigQuery project ID here
project_id = 'mgmt-467-471613' # Replace with your actual project ID

# Construct a BigQuery client object
client = bigquery.Client(project=project_id)

print(f"BigQuery client created with project: {client.project}")

BigQuery client created with project: mgmt-467-471613



## Task 5.0: Bucket a Continuous Feature

**🎯 Goal:** Group 'total_minutes' into categories: low, medium, high.  
**📌 Requirements:** Use CASE WHEN or IF statements to create 'watch_time_bucket'.

---

### 🧠 Prompt Template  
> Write SQL that creates a new column watch_time_bucket based on total_minutes thresholds (<100, 100–300, >300).

---

### 👩‍🏫 Example Prompt  
> Create a new column watch_time_bucket with values 'low', 'medium', or 'high' based on total_minutes.

---

### 🔍 Exploration  
How does churn rate vary across these buckets?


In [10]:
%%bigquery --project mgmt-467-471613
ALTER TABLE
    `mgmt-467-471613.netflix.users`
ADD COLUMN
    total_minutes INTEGER;

Query is running:   0%|          |

""


In [16]:
%%bigquery --project mgmt-467-471613
UPDATE
    `mgmt-467-471613.netflix.users` AS u
SET
    total_minutes = CAST(wh.total_minutes AS INT64)
FROM (
    SELECT
        user_id,
        SUM(watch_duration_minutes) AS total_minutes
    FROM
        `mgmt-467-471613.netflix.watch_history`
    GROUP BY
        user_id
) AS wh
WHERE
    u.user_id = wh.user_id;

Query is running:   0%|          |

""


In [17]:
%%bigquery --project mgmt-467-471613
SELECT
    *
FROM
    `mgmt-467-471613.netflix.users` -- Replace with your actual users table name
LIMIT 10;

Query is running:   0%|          |

Downloading:   0%|          |

,user_id,email,first_name,last_name,age,gender,country,state_province,city,subscription_plan,subscription_start_date,is_active,monthly_spend,primary_device,household_size,created_at,total_minutes
0,user_00015,barnesbrandy@example.net,Sarah,Santiago,45.0,Female,Canada,Alberta,West Randy,Basic,2023-12-16,False,5.14,Mobile,3.0,2022-08-13 06:39:47.240847+00:00,1591
1,user_00015,barnesbrandy@example.net,Sarah,Santiago,45.0,Female,Canada,Alberta,West Randy,Basic,2023-12-16,False,5.14,Mobile,3.0,2022-08-13 06:39:47.240847+00:00,1591
2,user_00015,barnesbrandy@example.net,Sarah,Santiago,45.0,Female,Canada,Alberta,West Randy,Basic,2023-12-16,False,5.14,Mobile,3.0,2022-08-13 06:39:47.240847+00:00,1591
3,user_00021,emurphy@example.com,Darlene,Frazier,38.0,Female,Canada,Alberta,North Natalieview,Basic,2024-04-29,True,17.24,Laptop,3.0,2022-09-13 23:45:07.830930+00:00,676
4,user_00021,emurphy@example.com,Darlene,Frazier,38.0,Female,Canada,Alberta,North Natalieview,Basic,2024-04-29,True,17.24,Laptop,3.0,2022-09-13 23:45:07.830930+00:00,676
5,user_00021,emurphy@example.com,Darlene,Frazier,38.0,Female,Canada,Alberta,North Natalieview,Basic,2024-04-29,True,17.24,Laptop,3.0,2022-09-13 23:45:07.830930+00:00,676
6,user_00041,michelle64@example.net,Chelsea,Meza,29.0,Female,Canada,Alberta,West Donna,Standard,2023-04-21,True,25.73,Laptop,2.0,2023-01-21 14:04:27.262786+00:00,1586
7,user_00041,michelle64@example.net,Chelsea,Meza,29.0,Female,Canada,Alberta,West Donna,Standard,2023-04-21,True,25.73,Laptop,2.0,2023-01-21 14:04:27.262786+00:00,1586
8,user_00041,michelle64@example.net,Chelsea,Meza,29.0,Female,Canada,Alberta,West Donna,Standard,2023-04-21,True,25.73,Laptop,2.0,2023-01-21 14:04:27.262786+00:00,1586
9,user_00042,enorris@example.com,Jesse,Warner,58.0,Male,Canada,Alberta,Lake Jesusview,Standard,2023-11-10,True,27.64,Laptop,6.0,2024-07-12 16:43:25.972535+00:00,289


In [39]:
%%bigquery --project mgmt-467-471613
ALTER TABLE
    `mgmt-467-471613.netflix.users` -- Replace with your actual users table name
ADD COLUMN
    watch_time_bucket STRING; -- Or an appropriate data type for the bucket names

Query is running:   0%|          |

""


In [40]:
%%bigquery --project mgmt-467-471613
UPDATE
    `mgmt-467-471613.netflix.users` -- Replace with your actual users table name
SET
    watch_time_bucket = CASE
        WHEN total_minutes < 100 THEN 'low'
        WHEN total_minutes BETWEEN 100 AND 300 THEN 'medium'
        WHEN total_minutes > 300 THEN 'high'
        ELSE 'unknown' -- Handle potential NULLs or other cases
    END
WHERE
    total_minutes IS NOT NULL; -- Optional: only update rows where total_minutes is not NULL

Query is running:   0%|          |

""



## Task 5.1: Create a Binary Flag Feature

**🎯 Goal:** Add a binary column flag_binge (1 if total_minutes > 500).  
**📌 Requirements:** Use IF logic to create a binary column in SQL.

---

### 🧠 Prompt Template  
> Write a SQL query that adds flag_binge = 1 if total_minutes > 500, else 0.

---

### 👩‍🏫 Example Prompt  
> Add a binary column flag_binge to identify binge-watchers.

---

### 🔍 Exploration  
Are binge-watchers more or less likely to churn?


In [22]:
%%bigquery --project mgmt-467-471613
ALTER TABLE
    `mgmt-467-471613.netflix.users` -- Replace with your actual users table name
ADD COLUMN
    flag_binge INT64; -- Or BOOLEAN, depending on your preference for binary flags

Query is running:   0%|          |

""


In [23]:
%%bigquery --project mgmt-467-471613
UPDATE
    `mgmt-467-471613.netflix.users` -- Replace with your actual users table name
SET
    flag_binge = CASE
        WHEN total_minutes > 500 THEN 1
        ELSE 0
    END
WHERE
    total_minutes IS NOT NULL; -- Optional: only update rows where total_minutes is not NULL

Query is running:   0%|          |

""



## Task 5.2: Create an Interaction Term

**🎯 Goal:** Create plan_region_combo by combining plan_tier and region.  
**📌 Requirements:** Use CONCAT or STRING functions.

---

### 🧠 Prompt Template  
> Generate SQL to create a new column by combining plan_tier and region with an underscore.

---

### 👩‍🏫 Example Prompt  
> Create a column called plan_region_combo as CONCAT(plan_tier, '_', region).

---

### 🔍 Exploration  
Which plan-region combos have highest churn?


In [24]:
%%bigquery --project mgmt-467-471613
SELECT
    subscription_plan,
    state_province,
    CONCAT(subscription_plan, '_', state_province) AS plan_region_combo
FROM
    `mgmt-467-471613.netflix.users`; -- Replace with your actual table name

Query is running:   0%|          |

Downloading:   0%|          |

,subscription_plan,state_province,plan_region_combo
0,Basic,Alberta,Basic_Alberta
1,Basic,Alberta,Basic_Alberta
2,Basic,Alberta,Basic_Alberta
3,Basic,Alberta,Basic_Alberta
4,Basic,Alberta,Basic_Alberta
...,...,...,...
30895,Standard,Wisconsin,Standard_Wisconsin
30896,Standard,Wisconsin,Standard_Wisconsin
30897,Premium+,Wisconsin,Premium+_Wisconsin
30898,Premium+,Wisconsin,Premium+_Wisconsin


In [25]:
%%bigquery --project mgmt-467-471613
SELECT
    CONCAT(subscription_plan, '_', state_province) AS plan_region_combo,
    COUNT(*) AS total_users,
    SUM(CASE WHEN is_active = FALSE THEN 1 ELSE 0 END) AS churned_users,
    SAFE_DIVIDE(SUM(CASE WHEN is_active = FALSE THEN 1 ELSE 0 END), COUNT(*)) AS churn_rate
FROM
    `mgmt-467-471613.netflix.users` -- Replace with your actual table name
GROUP BY
    plan_region_combo
ORDER BY
    churn_rate DESC; -- Order by churn rate in descending order to see the highest churn combos first

Query is running:   0%|          |

Downloading:   0%|          |

,plan_region_combo,total_users,churned_users,churn_rate
0,Premium+_Michigan,120,33,0.275000
1,Premium+_North Carolina,114,30,0.263158
2,Premium+_Indiana,93,24,0.258065
3,Premium+_Quebec,78,18,0.230769
4,Basic_Ohio,159,36,0.226415
...,...,...,...,...
115,Premium+_Maryland,96,6,0.062500
116,Premium+_British Columbia,96,6,0.062500
117,Basic_Prince Edward Island,162,9,0.055556
118,Premium+_Arizona,114,6,0.052632



## Task 5.3: Add Missingness Indicator Flags

**🎯 Goal:** Add binary flags to capture NULL values in age_band and avg_rating.  
**📌 Requirements:** Use IS NULL logic to create new flag columns.

---

### 🧠 Prompt Template  
> Create a new column is_missing_[col_name] that is 1 when column is NULL, else 0.

---

### 👩‍🏫 Example Prompt  
> Add is_missing_age that flags rows where age_band IS NULL.

---

### 🔍 Exploration  
Do missing values correlate with churn?


In [26]:
%%bigquery --project mgmt-467-471613
ALTER TABLE
    `mgmt-467-471613.netflix.users` -- Replace with your actual users table name
ADD COLUMN
    is_missing_age INT64; -- Or BOOLEAN

Query is running:   0%|          |

""


In [27]:
%%bigquery --project mgmt-467-471613
UPDATE
    `mgmt-467-471613.netflix.users` -- Replace with your actual users table name
SET
    is_missing_age = CASE
        WHEN age IS NULL THEN 1
        ELSE 0
    END
WHERE
    age IS NOT NULL OR age IS NULL; -- This condition effectively updates all rows

Query is running:   0%|          |

""


In [29]:
%%bigquery --project mgmt-467-471613
SELECT
    age,
    is_missing_age
FROM
    `mgmt-467-471613.netflix.users` -- Replace with your actual users table name
LIMIT 10; -- Adjust limit as needed

Query is running:   0%|          |

Downloading:   0%|          |

,age,is_missing_age
0,45.0,0
1,45.0,0
2,45.0,0
3,38.0,0
4,38.0,0
5,38.0,0
6,29.0,0
7,29.0,0
8,29.0,0
9,58.0,0


In [30]:
%%bigquery --project mgmt-467-471613
SELECT
    is_missing_age,
    COUNT(*) AS total_users,
    SUM(CASE WHEN is_active = FALSE THEN 1 ELSE 0 END) AS churned_users,
    SAFE_DIVIDE(SUM(CASE WHEN is_active = FALSE THEN 1 ELSE 0 END), COUNT(*)) AS churn_rate
FROM
    `mgmt-467-471613.netflix.users` -- Replace with your actual table name
GROUP BY
    is_missing_age;

Query is running:   0%|          |

Downloading:   0%|          |

,is_missing_age,total_users,churned_users,churn_rate
0,0,27213,4029,0.148054
1,1,3687,543,0.147274



## Task 5.4: Create Time-Based Features (Optional)

**🎯 Goal:** Add a column days_since_last_login.  
**📌 Requirements:** Use DATE_DIFF with CURRENT_DATE and last_login_date.

---

### 🧠 Prompt Template  
> Write SQL to create a column showing days since last login using DATE_DIFF.

---

### 👩‍🏫 Example Prompt  
> Add a column days_since_last_login = DATE_DIFF(CURRENT_DATE(), last_login_date, DAY).

---

### 🔍 Exploration  
Does login recency affect churn rate?


In [31]:
## Query the watch_history table to find the maximum watch_date for each user_id.
%%bigquery --project mgmt-467-471613
SELECT
    user_id,
    MAX(watch_date) AS last_watch_date
FROM
    `mgmt-467-471613.netflix.watch_history`
GROUP BY
    user_id;

Query is running:   0%|          |

Downloading:   0%|          |

,user_id,last_watch_date
0,user_00074,2025-11-22
1,user_03470,2025-10-29
2,user_04285,2025-11-19
3,user_09560,2025-07-28
4,user_09343,2025-05-20
...,...,...
9995,user_08979,2025-10-03
9996,user_05053,2025-11-30
9997,user_07782,2025-12-15
9998,user_06995,2025-08-13


In [32]:
## Add a `last_login_date` column to the `users` table
%%bigquery --project mgmt-467-471613
ALTER TABLE
    `mgmt-467-471613.netflix.users`
ADD COLUMN
    last_login_date DATE;

Query is running:   0%|          |

""


In [33]:
## Update the `last login date` column in the `users` table with the most recent watch date found in the previous step.
%%bigquery --project mgmt-467-471613
UPDATE
    `mgmt-467-471613.netflix.users` AS u
SET
    last_login_date = wh.last_watch_date
FROM (
    SELECT
        user_id,
        MAX(watch_date) AS last_watch_date
    FROM
        `mgmt-467-471613.netflix.watch_history`
    GROUP BY
        user_id
) AS wh
WHERE
    u.user_id = wh.user_id;

Query is running:   0%|          |

""


In [34]:
## Add a new column named days_since_last_login to the users table in BigQuery. This column will store the calculated number of days between the current date and the user's last login date.
%%bigquery --project mgmt-467-471613
ALTER TABLE
    `mgmt-467-471613.netflix.users`
ADD COLUMN
    days_since_last_login INT64;

Query is running:   0%|          |

""


In [35]:
## Calculate and update the days since last login column in the users table using the difference in days between the current date and the last_login_date.
%%bigquery --project mgmt-467-471613
UPDATE
    `mgmt-467-471613.netflix.users`
SET
    days_since_last_login = DATE_DIFF(CURRENT_DATE(), last_login_date, DAY)
WHERE
    last_login_date IS NOT NULL;

Query is running:   0%|          |

""


In [36]:
## Select a few rows from the users table to ensure the last_login_date and days_since_last_login columns have been populated correctly.
%%bigquery --project mgmt-467-471613
SELECT
    user_id,
    last_login_date,
    days_since_last_login
FROM
    `mgmt-467-471613.netflix.users`
LIMIT 10;

Query is running:   0%|          |

Downloading:   0%|          |

,user_id,last_login_date,days_since_last_login
0,user_00186,2025-12-31,-67
1,user_00186,2025-12-31,-67
2,user_00186,2025-12-31,-67
3,user_00196,2025-12-31,-67
4,user_00196,2025-12-31,-67
5,user_00196,2025-12-31,-67
6,user_00221,2025-12-31,-67
7,user_00221,2025-12-31,-67
8,user_00221,2025-12-31,-67
9,user_00247,2025-12-31,-67


In [37]:
## Explore correlation with churn
%%bigquery --project mgmt-467-471613
SELECT
    CASE
        WHEN days_since_last_login IS NULL THEN 'No Login Data'
        WHEN days_since_last_login <= 30 THEN '0-30 Days'
        WHEN days_since_last_login > 30 AND days_since_last_login <= 90 THEN '31-90 Days'
        WHEN days_since_last_login > 90 AND days_since_last_login <= 180 THEN '91-180 Days'
        ELSE '>180 Days'
    END AS login_recency_bucket,
    COUNT(*) AS total_users,
    SUM(CASE WHEN is_active = FALSE THEN 1 ELSE 0 END) AS churned_users,
    SAFE_DIVIDE(SUM(CASE WHEN is_active = FALSE THEN 1 ELSE 0 END), COUNT(*)) AS churn_rate
FROM
    `mgmt-467-471613.netflix.users`
GROUP BY
    login_recency_bucket
ORDER BY
    churn_rate DESC;

Query is running:   0%|          |

Downloading:   0%|          |

,login_recency_bucket,total_users,churned_users,churn_rate
0,>180 Days,1095,186,0.169863
1,0-30 Days,22887,3420,0.149430
2,31-90 Days,4326,612,0.141470
3,91-180 Days,2592,354,0.136574


## Summary:

### Data Analysis Key Findings

*   Users who have not logged in for more than 180 days exhibit the highest churn rate (16.99%).
*   Users who have logged in within the last 30 days show the second highest churn rate (14.94%).

### Insights or Next Steps

*   Further investigate the churn behavior of recently active users (0-30 days) to understand why their churn rate is relatively high compared to users with moderate login recency.
*   Consider using the `days_since_last_login` feature in a churn prediction model.



## Task 5.5: Assemble Enhanced Feature Table

**🎯 Goal:** Create churn_features_enhanced with all engineered columns.  
**📌 Requirements:** Include all prior features + engineered columns.

---

### 🧠 Prompt Template  
> Generate SQL to create churn_features_enhanced with new columns: watch_time_bucket, plan_region_combo, flag_binge, etc.

---

### 👩‍🏫 Example Prompt  
> Build a new table churn_features_enhanced with all original features + engineered ones.

---

### 🔍 Exploration  
Are row counts stable? Any NULLs introduced?


In [41]:
%%bigquery --project mgmt-467-471613
CREATE TABLE
    `mgmt-467-471613.netflix.churn_features_enhanced` AS
SELECT
    cf.*, -- Select all columns from the original churn_features table
    u.watch_time_bucket,
    u.flag_binge,
    u.is_missing_age,
    CONCAT(u.subscription_plan, '_', u.state_province) AS plan_region_combo,
    u.days_since_last_login
FROM
    `mgmt-467-471613.netflix.churn_features` AS cf -- Replace with your actual churn_features table name
JOIN
    `mgmt-467-471613.netflix.users` AS u -- Replace with your actual users table name
ON
    cf.user_id = u.user_id;

Query is running:   0%|          |

""


In [42]:
%%bigquery --project mgmt-467-471613
SELECT
    *
FROM
    `mgmt-467-471613.netflix.churn_features_enhanced` -- Replace with your actual table name
LIMIT 10; -- Adjust the limit as needed to see the columns and some data

Query is running:   0%|          |

Downloading:   0%|          |

,user_id,state_province,subscription_plan,age,avg_rating,total_minutes,churn_label,watch_time_bucket,flag_binge,is_missing_age,plan_region_combo,days_since_last_login
0,user_00015,Alberta,Basic,45.0,4.0,28630.8,False,high,1,0,Basic_Alberta,68
1,user_00015,Alberta,Basic,45.0,4.0,28630.8,False,high,1,0,Basic_Alberta,68
2,user_00015,Alberta,Basic,45.0,4.0,28630.8,False,high,1,0,Basic_Alberta,68
3,user_00021,Alberta,Basic,38.0,3.0,6085.8,True,high,1,0,Basic_Alberta,248
4,user_00021,Alberta,Basic,38.0,3.0,6085.8,True,high,1,0,Basic_Alberta,248
5,user_00021,Alberta,Basic,38.0,3.0,6085.8,True,high,1,0,Basic_Alberta,248
6,user_00587,Alberta,Basic,33.0,NaN,11509.2,True,high,1,0,Basic_Alberta,-54
7,user_00587,Alberta,Basic,33.0,NaN,11509.2,True,high,1,0,Basic_Alberta,-54
8,user_00587,Alberta,Basic,33.0,NaN,11509.2,True,high,1,0,Basic_Alberta,-54
9,user_00587,Alberta,Basic,33.0,NaN,11509.2,True,high,1,0,Basic_Alberta,-54



## Task 6: Retrain Model on Engineered Features

**🎯 Goal:** Train a logistic regression model using churn_features_enhanced.  
**📌 Requirements:** Use BQML logistic_reg model with new feature columns.

---

### 🧠 Prompt Template  
> Write CREATE MODEL SQL using enhanced features including flags and buckets.

---

### 👩‍🏫 Example Prompt  
> Retrain churn_model_enhanced using watch_time_bucket, flag_binge, plan_region_combo.

---

### 🔍 Exploration  
Does model accuracy improve?

Yes, it does improve the accuracy


In [43]:
%%bigquery --project mgmt-467-471613
CREATE OR REPLACE MODEL
    `mgmt-467-471613.netflix.churn_model_enhanced` -- Replace with your actual dataset and model name
OPTIONS (
    model_type = 'LOGISTIC_REG',
    input_label_cols = ['churn_label'] -- Replace with your actual churn label column name
) AS
SELECT
    * EXCEPT(user_id) -- Exclude user_id or any other non-feature columns
FROM
    `mgmt-467-471613.netflix.churn_features_enhanced`; -- Replace with your actual enhanced features table name

Query is running:   0%|          |

""


In [44]:
%%bigquery --project mgmt-467-471613
SELECT
    *
FROM
    ML.EVALUATE(MODEL `mgmt-467-471613.netflix.churn_model_enhanced`, -- Replace with your actual dataset and model name
        (
            SELECT
                *
            FROM
                `mgmt-467-471613.netflix.churn_features_enhanced` -- Replace with your actual enhanced features table name
        )
    );

Query is running:   0%|          |

Downloading:   0%|          |

,precision,recall,accuracy,f1_score,log_loss,roc_auc
0,0.852039,1.0,0.852039,0.920109,0.414721,0.577082



## Task 7: Compare Model Performance

**🎯 Goal:** Compare base model vs enhanced model using ML.EVALUATE.  
**📌 Requirements:** Use same evaluation query for both models.

---

### 🧠 Prompt Template  
> Write a SQL query to evaluate churn_model_enhanced and compare with churn_model.

---

### 👩‍🏫 Example Prompt  
> Compare ML.EVALUATE output from both models side-by-side.

---

### 🔍 Exploration  
Which features made the most difference?

I think the flag_binge made the biggest difference because it has the largest abosolute weight out of all the features.


In [45]:
%%bigquery --project mgmt-467-471613
SELECT
    *
FROM
    ML.EVALUATE(MODEL `mgmt-467-471613.netflix.churn_model_enhanced`, -- Replace with your actual dataset and enhanced model name
        (
            SELECT
                *
            FROM
                `mgmt-467-471613.netflix.churn_features_enhanced` -- Replace with your actual enhanced features table name
        )
    );

Query is running:   0%|          |

Downloading:   0%|          |

,precision,recall,accuracy,f1_score,log_loss,roc_auc
0,0.852039,1.0,0.852039,0.920109,0.414721,0.577082


**Evaluate original churn_model:**

In [46]:
%%bigquery --project mgmt-467-471613
SELECT
    *
FROM
    ML.EVALUATE(MODEL `mgmt-467-471613.netflix.churn_model`, -- Replace with your actual dataset and original model name
        (
            SELECT
                *
            FROM
                `mgmt-467-471613.netflix.churn_features` -- Replace with your actual original features table name
        )
    );

Query is running:   0%|          |

Downloading:   0%|          |

,precision,recall,accuracy,f1_score,log_loss,roc_auc
0,0.8519,1.0,0.8519,0.920028,0.418237,0.540115


In [47]:
%%bigquery --project mgmt-467-471613
SELECT
    *
FROM
    ML.WEIGHTS(MODEL `mgmt-467-471613.netflix.churn_model`); -- Replace with your actual dataset and model name

Query is running:   0%|          |

Downloading:   0%|          |

,processed_input,weight,category_weights
0,state_province,NaN,"[{'category': 'Illinois', 'weight': 0.58775151..."
1,subscription_plan,NaN,"[{'category': 'Premium', 'weight': 0.581808863..."
2,age,1.166129e-03,[]
3,avg_rating,4.597836e-02,[]
4,total_minutes,-5.370469e-08,[]
5,__INTERCEPT__,3.856270e-01,[]


In [49]:
%%bigquery --project mgmt-467-471613
SELECT
    *
FROM
    ML.WEIGHTS(MODEL `mgmt-467-471613.netflix.churn_model_enhanced`); -- Replace with your actual dataset and model name

Query is running:   0%|          |

Downloading:   0%|          |

,processed_input,weight,category_weights
0,state_province,NaN,"[{'category': 'Virginia', 'weight': 0.40256260..."
1,subscription_plan,NaN,"[{'category': 'Premium', 'weight': 0.343102868..."
2,age,1.552001e-03,[]
3,avg_rating,3.938721e-02,[]
4,total_minutes,9.528370e-08,[]
5,watch_time_bucket,NaN,"[{'category': 'high', 'weight': 0.344723402159..."
6,flag_binge,-1.520868e-01,[]
7,is_missing_age,5.136352e-02,[]
8,plan_region_combo,NaN,"[{'category': 'Premium_Nova Scotia', 'weight':..."
9,days_since_last_login,7.898184e-05,[]
